In [ ]:
from utils.WikipediaDownload import WikipediaDownload

# download the wikipedia dump for a specific year / month
# CAREFUL! successive executions could lead to an ip-block. wikimedia only allows a few downloads at once
wd = WikipediaDownload(year=2026, month=4, language='de', output_path='./downloads')
wd.download_files()

In [ ]:
from utils.ArticleExtract import Extract, ExtractorConfig
from utils.DB import DB
from utils.Stopword import Stopword

# extract and clean articles
db = DB()
db.connect(path='./db')
db.delete()

compress_size = 200 * 1024 * 1024

columns = [['id', 'int'], ['title', 'text'], ['section_title', 'text']]
db.create_file(table_name='tabelle1', columns=columns, base_path='./db', compress=True, compress_size=compress_size)

config = ExtractorConfig.get_default_config()
extr = Extract('./downloads', extractor_config=config)

sw = Stopword()

store_func = lambda m, t: db.stream_insert_compr(m, t, lambda n, s: s)
extr.extract_pages(store_func=store_func, text_transform_func=sw.text_to_bytes, limit=None)

db.stream_insert_finish()

In [ ]:
from utils.DB import DB

db = DB()
db.connect(path='./db')
data, text, size = db.query_compr(max_size=200 * 1024 * 1024)

for i in range(10):
    print(data[i], text[data[i][0]:data[i][1]])

(0, 6444, 2163201, '1752', 'Ereignisse') b'   * 18. m\x80rz: francesco loredan wird elf tage nach dem tod seines vorg\x80ngers pietro grimani zum dogen von venedig gew\x80hlt. zum zeitpunkt seiner wahl ist der reichtum der familie loredan in venedig legend\x80r, sie besitzt mehrere pal\x80ste, wie den palazzo di san stefano, den palazzo loredan-cini, den palazzo loredan/farsetti und die c\xe1 loredan am canal grande, die francesco dem \x81sterreichischen botschafter gegen vorkasse der gesamtmiete f\x82r 29 jahre und die verpflichtung zu restauration und unterhalt des palastes vermietet hat. der neue doge f\x82hrt ein luxuri\x81ses leben mit seinem bruder und seiner schw\x80gerin im dogenpalast, f\x82hrt die republik aber auch zu wirtschaftlicher bl\x82te.  * k\x81nig friedrich ii. von preu\x83en schreibt sein politisches testament nieder.  * juli: michel-ange duquesne de menneville wird als nachfolger des am 17. m\x80rz verstorbenen jacques-pierre de taffanel de la jonqui\xe9re general

In [8]:
import sqlite3
import zlib
import datetime

con = sqlite3.connect('./db/database.db')
block = con.execute('select * from blocks limit 1').fetchone()

block_id, start, end, num_entrys, size = block

with open('./db/database.bin', 'rb') as fd:
    data = fd.read(end)
start_time = datetime.datetime.now()
decompr = zlib.decompress(data)
end_time = datetime.datetime.now()

print(f'decompression of {size} byte block: {(end_time - start_time).total_seconds() * 1000} ms')

decompression of 209714346 byte block: 511.037 ms
